# 📖 Notebook 5: Observability — Monitoring with Prometheus and Grafana

You can't fix what you can't see. In this notebook you'll install a production-grade monitoring stack
inside your minikube cluster and learn how to collect metrics, build dashboards, and create alerts.

We'll use the same tools that Netflix, Spotify, and most Fortune 500 companies rely on:
**Prometheus** for collecting metrics and **Grafana** for visualizing them.

> **Prerequisites: Notebooks 01–03.** This notebook needs the `k8s-lab` namespace with
> its three deployments *and* the Services from `../manifests/service.yaml` — Prometheus
> discovers scrape targets through Services, not Deployments.
>
> **Resources**: kube-prometheus-stack is the heaviest thing in this series. Prometheus
> alone requests 400 Mi and 200m CPU (trimmed down from the chart's defaults in
> `../manifests/prometheus-values.yaml`), and node-exporter runs on every node. On a
> 2 CPU / 4 GB minikube the pods will sit in `Pending`. Notebook 01 sizes the cluster at
> `--cpus=4 --memory=6144` for exactly this reason.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['minikube', 'kubectl', 'helm']
INSTALL_HINTS = {
    'minikube': 'https://minikube.sigs.k8s.io/docs/start/  (or `brew install minikube`)',
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
    'helm': 'https://helm.sh/docs/intro/install/  (or `brew install helm`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
# A `!kubectl`/`!helm` line that exits non-zero prints red text but does not fail
# the cell, so everything this notebook claims is also asserted in Python. The
# Prometheus helpers below query the HTTP API directly -- clicking around the UI
# proves nothing to a reader of the executed notebook.
import json
import socket
import subprocess
import time
import urllib.error
import urllib.request

NS = "k8s-lab"


def kget(*args, ns=NS):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def wait_until(predicate, timeout=170, interval=10, what="condition"):
    """Poll until predicate() is truthy. Returns its value, or None on timeout.

    Notebook runners cap how long one cell may block (180s is common), so long
    waits are written as a bounded poll that the next cell can continue, rather
    than a single `--wait --timeout 10m` that blows the budget."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    return None


# The kube-prometheus-stack chart installs three Deployments directly, and the
# prometheus-operator then creates two StatefulSets from the Prometheus and
# Alertmanager custom resources. Listing them by NAME matters: a check that just
# counts "all workloads in the namespace" passes at the moment only the three
# Deployments exist, before the operator has created the other two.
MONITORING_WORKLOADS = {
    "deployment": ("prometheus-grafana",
                   "prometheus-kube-prometheus-operator",
                   "prometheus-kube-state-metrics"),
    "statefulset": ("alertmanager-prometheus-kube-prometheus-alertmanager",
                    "prometheus-prometheus-kube-prometheus-prometheus"),
}


def workloads_ready(ns, expected):
    """{name: ready replica count} for the named workloads in a namespace."""
    counts = {}
    for kind, names in expected.items():
        for name in names:
            r = subprocess.run(["kubectl", "get", kind, name, "-n", ns, "-o", "json"],
                               capture_output=True, text=True)
            counts[name] = (json.loads(r.stdout)["status"].get("readyReplicas", 0)
                            if r.returncode == 0 else 0)
    return counts


def port_is_free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def start_port_forward(target, remote_port, preferred, ns, attempts=4):
    """Start `kubectl port-forward` and return (process, local_port).

    The preferred port is used when it is free. Hard-coding 9090 is the usual
    thing to do and the usual thing to regret: if anything else on your laptop
    already holds it, port-forward exits immediately and every later request
    fails with a connection error that looks like a Prometheus problem. So we
    fall back to an ephemeral port -- and retry, because another process can
    still claim it in the moment between our probe and kubectl's bind."""
    last_error = ""
    for attempt in range(attempts):
        port = preferred if (attempt == 0 and port_is_free(preferred)) else 0
        if port == 0:
            with socket.socket() as s:
                s.bind(("127.0.0.1", 0))
                port = s.getsockname()[1]
        proc = subprocess.Popen(
            ["kubectl", "port-forward", "-n", ns, target, f"{port}:{remote_port}"],
            stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        deadline = time.time() + 20
        while time.time() < deadline:
            if proc.poll() is not None:
                last_error = (proc.stderr.read() or "").strip()[:200]
                break
            if not port_is_free(port):
                if port != preferred:
                    print(f"(port {preferred} was busy -- using {port} instead)")
                return proc, port
            time.sleep(0.5)
        proc.terminate()
        last_error = last_error or "port never started listening"
    raise RuntimeError(f"port-forward to {target} never came up: {last_error}")


def prom_api(path, base=None, **params):
    """GET one of Prometheus's HTTP API endpoints and return the `data` field."""
    url = f"{base or PROM}/api/v1/{path}"
    if params:
        url += "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=15) as r:
        body = json.loads(r.read().decode())
    assert body["status"] == "success", body
    return body["data"]


import urllib.parse  # noqa: E402  (kept next to its only user)

print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the three pillars of observability: metrics, logs, and traces
- Enable **Metrics Server** and use `kubectl top` to inspect resource usage
- Install **Prometheus + Grafana** via Helm (kube-prometheus-stack)
- Query Prometheus for built-in Kubernetes metrics
- Create a **ServiceMonitor** so Prometheus scrapes your own app's `/metrics` endpoint
- Access **Grafana** dashboards to visualize cluster and application health
- Write a **PrometheusRule** alert that fires when a service goes down

## 🛠️ Setup

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

Make sure minikube is running, the sample apps from Notebook 02 are deployed in the
`k8s-lab` namespace, and the Services from Notebook 03 exist.

In [ ]:
# Verify cluster is running and apps are deployed
!minikube status
!echo '---'
!kubectl get deployments -n k8s-lab
!echo '---'
# Prometheus discovers targets via Services, so these must exist too (notebook 03).
!kubectl get svc -n k8s-lab

## 🔭 The Three Pillars of Observability

When something goes wrong in production, you need three types of data to find the problem:

| Pillar | What It Tells You | Example Tool |
|--------|-------------------|-------------|
| **Metrics** | Numbers over time — CPU, memory, request count, error rate | Prometheus |
| **Logs** | Detailed text records of what happened | Loki, ELK |
| **Traces** | The path a request takes across multiple services | Jaeger, Tempo |

In this lab we focus on **Metrics** — the most important pillar for keeping a cluster healthy.

```
Your App ──▶ /metrics endpoint ──▶ Prometheus (scrapes every 30s) ──▶ Grafana (dashboards)
                                         │
                                         ▼
                                   AlertManager ──▶ Slack / PagerDuty
```

## 📏 Step 1: Enable Metrics Server

**Metrics Server** is a lightweight component that collects CPU and memory usage from every node and pod.
It powers two things:
- `kubectl top` — see resource usage from the command line
- **HPA** (Horizontal Pod Autoscaler) — automatically scale pods based on CPU/memory

Minikube makes it easy — just enable the addon:

In [ ]:
# Enable metrics server (idempotent -- re-enabling an enabled addon is a no-op)
!minikube addons enable metrics-server

# Wait for it to be ready (takes ~30 seconds)
!kubectl wait --for=condition=ready pod -l k8s-app=metrics-server -n kube-system --timeout=180s

print()
# metrics-server is not an ordinary Deployment you query directly: it registers
# itself as an APIService, which is how `kubectl top` and the HPA controller can
# ask the normal Kubernetes API for usage numbers.
!kubectl get apiservice v1beta1.metrics.k8s.io

In [ ]:
# Now we can see resource usage!
# Node-level: how much CPU and memory is the whole node using?
!kubectl top nodes

print()

# Pod-level: which pods are using the most resources?
!kubectl top pods -n k8s-lab

# `kubectl top` needs two scrape intervals before metrics-server can report a
# rate, so it answers "Metrics API not available" for the first ~60 seconds.
# Poll rather than assume -- the HPA in notebook 10 depends on this same API.
deadline = time.time() + 180
while time.time() < deadline:
    r = subprocess.run(["kubectl", "top", "pods", "-n", NS, "--no-headers"],
                       capture_output=True, text=True)
    rows = [l for l in r.stdout.strip().splitlines() if l.strip()]
    if r.returncode == 0 and rows:
        break
    time.sleep(5)
else:
    raise AssertionError("metrics-server never served pod metrics; `kubectl top` and "
                         "every `type: Resource` HPA would be dead in this cluster")
print(f"\n✅ metrics.k8s.io is serving usage for {len(rows)} pods in {NS}")

**What you see**: CPU in millicores (1000m = 1 full CPU core) and Memory in Mi/Gi.

### How the two metric paths differ — this trips everyone up

There are two completely separate metrics pipelines in a Kubernetes cluster and they do
not talk to each other:

| | metrics-server | Prometheus |
|---|---|---|
| Serves | the `metrics.k8s.io` **Kubernetes API** | its own HTTP API and PromQL |
| Consumers | `kubectl top`, the **HPA controller**, VPA | Grafana, alerting, you |
| Retention | none — only the latest sample, in memory | a time-series database on disk |
| Source | the kubelet's `/metrics/resource` on each node | scraping every `/metrics` endpoint it is told about |

⚠️ Installing Prometheus does **not** give you `kubectl top` and does **not** make a
plain CPU-based HPA work. An HPA with `type: Resource` reads `metrics.k8s.io` and nothing
else, so a cluster without metrics-server shows `TARGETS: <unknown>/60%` forever. (Making
an HPA scale on a *Prometheus* metric is possible, but it needs a separate
`prometheus-adapter` that republishes those metrics under `custom.metrics.k8s.io`.)

`kubectl top` returning `error: Metrics API not available` for the first ~60 seconds
after enabling the addon is normal — metrics-server needs two scrape intervals before it
can report a rate.

## 📊 Step 2: Install Prometheus + Grafana

The **kube-prometheus-stack** Helm chart installs everything you need in one command:

```
┌──────────────────────────────────────────────────────────────┐
│  kube-prometheus-stack (Helm chart)                          │
│                                                              │
│  ┌─────────────┐  ┌──────────────┐  ┌───────────────────┐  │
│  │ Prometheus   │  │ Grafana       │  │ AlertManager      │  │
│  │ (collects    │  │ (visualizes   │  │ (sends alerts     │  │
│  │  metrics)    │  │  dashboards)  │  │  to Slack, etc.)  │  │
│  └──────┬──────┘  └──────────────┘  └───────────────────┘  │
│         │                                                    │
│  ┌──────▼──────┐  ┌──────────────┐                          │
│  │ node-exporter│  │kube-state-   │                          │
│  │ (OS metrics) │  │metrics       │                          │
│  │              │  │(K8s objects) │                          │
│  └─────────────┘  └──────────────┘                          │
└──────────────────────────────────────────────────────────────┘
```

- **Prometheus** scrapes `/metrics` endpoints from all pods every 30 seconds
- **node-exporter** exposes OS-level metrics (disk, network, CPU per core)
- **kube-state-metrics** exposes Kubernetes object states (how many pods are running, pending, failed)
- **Grafana** provides pre-built dashboards for all of the above
- **AlertManager** routes alerts to your notification channels

In [ ]:
# Without --force-update, `helm repo add` fails on a second run with
# "repository name (prometheus-community) already exists".
!helm repo add prometheus-community https://prometheus-community.github.io/helm-charts --force-update
!helm repo update

In [ ]:
# `helm install` is not re-runnable; `helm upgrade --install` is.
#
# Deliberately WITHOUT `--wait`. These are big images (Prometheus, Grafana,
# Alertmanager, kube-state-metrics, node-exporter) and on a fresh cluster the
# pulls take minutes -- longer than a notebook runner lets a single cell block.
# Helm returns as soon as the objects are applied; the next two cells wait for
# the pods, with a budget each.
!helm upgrade --install prometheus prometheus-community/kube-prometheus-stack \
    -n monitoring --create-namespace \
    -f ../manifests/prometheus-values.yaml \
    --timeout 10m

print()
!kubectl get deploy,statefulset -n monitoring

In [ ]:
# First half of the wait. Nothing is asserted here -- the cell after this one
# keeps waiting and then checks.
def monitoring_up():
    counts = workloads_ready("monitoring", MONITORING_WORKLOADS)
    print("  ready:", {k.replace("prometheus-kube-prometheus-", "")
                        .replace("prometheus-", ""): v for k, v in counts.items()})
    return counts if all(v >= 1 for v in counts.values()) else None


print("pulling images and starting pods (this is the slow part of the series):")
state = monitoring_up() or wait_until(monitoring_up, timeout=160, interval=10,
                                      what="the monitoring stack")
print("all up" if state else "still starting -- the next cell keeps waiting")

In [ ]:
state = monitoring_up() or wait_until(monitoring_up, timeout=170, interval=10,
                                     what="the monitoring stack")

# Check all the pods that were created
!kubectl get pods -n monitoring

assert state, (
    "the kube-prometheus-stack pods never all became Ready. On a 6 GB cluster the "
    "usual cause is memory: `kubectl get pods -n monitoring` will show Pending, and "
    "`kubectl describe` the reason. Otherwise it is a slow image pull -- re-run this "
    "cell to keep waiting."
)
print(f"\n✅ Prometheus stack installed: {len(state)} workloads Ready")
print("You should see: prometheus, grafana, alertmanager, node-exporter, kube-state-metrics")

## 🔍 Step 3: Query Prometheus

Prometheus has its own query language called **PromQL**. Let's access it.

We'll use `kubectl port-forward` to access the Prometheus web UI from our browser.

In [ ]:
# Start a port-forward in the background so we can reach Prometheus. `!kubectl
# port-forward ... &` would NOT work: `!` runs in a subshell that is torn down at
# the end of the cell, taking the background job with it. subprocess.Popen is
# owned by the kernel and survives until we terminate it.
prom_proc, prom_port = start_port_forward(
    "svc/prometheus-kube-prometheus-prometheus", 9090, preferred=9090, ns="monitoring")
PROM = f"http://localhost:{prom_port}"

# Prove the connection works instead of assuming it: `buildinfo` is the cheapest
# endpoint Prometheus serves.
info = prom_api("status/buildinfo")
print(f"✅ Prometheus {info['version']} reachable at: {PROM}")
print("   Open it in your browser and try these PromQL queries:")
print()
print('   1. up                                    — which targets are being scraped?')
print('   2. kube_pod_status_phase{namespace="k8s-lab"}  — pod phases in our namespace')
print('   3. container_cpu_usage_seconds_total       — CPU usage by container')
print('   4. node_memory_MemAvailable_bytes          — available memory on the node')

# The same query, run for real, so the notebook demonstrates PromQL rather than
# describing it.
targets = prom_api("query", query="count(up == 1)")["result"]
print(f"\nPromQL `count(up == 1)` -> {targets[0]['value'][1]} healthy scrape targets")
assert int(targets[0]["value"][1]) > 0, "Prometheus is scraping nothing at all"

### 💡 Common PromQL Queries

| Query | What It Shows |
|-------|---------------|
| `up` | All scrape targets and whether they're reachable |
| `rate(container_cpu_usage_seconds_total[5m])` | CPU usage rate over 5 minutes |
| `container_memory_working_set_bytes` | Current memory usage per container |
| `kube_deployment_status_replicas_available` | How many replicas are running per deployment |
| `sum(rate(http_requests_total[5m])) by (service)` | Request rate per service |

## 🎯 Step 4: Expose Your App's Metrics

Our sample apps already expose a `/metrics` endpoint (check `apps/user-service/app.py`).
But Prometheus doesn't know about it yet — we need to tell it to scrape our app.

In the kube-prometheus-stack, you do this by creating a **ServiceMonitor** — a custom resource
that tells Prometheus: "hey, scrape this Service's pods on this port and path."

```
ServiceMonitor ──▶ selects a Service by label ──▶ Prometheus scrapes that Service's
                                                   endpoints on the named port at /metrics
```

### Two details that make or break a ServiceMonitor

**1. `spec.selector` matches the *Service*'s labels, not the pods'.** A ServiceMonitor is
a query over Services. It then scrapes the Endpoints behind whichever Services it matched.
Our Services in `../manifests/service.yaml` carry `labels: {app: user-service}`, so
`selector.matchLabels: {app: user-service}` is right — but for the Service's own labels,
which happen to be the same as the pods' here.

**2. `endpoints[].port` is the *name* of a Service port — never a number.** Writing
`port: "8001"` looks reasonable and is silently wrong: it is compared against
`Service.spec.ports[].name`, no port is named `"8001"`, so the ServiceMonitor matches the
Service and then produces **zero** scrape targets. Nothing errors; the target simply never
appears in Prometheus. (This is why Notebook 03's manifests name every Service port
`http`.) If a port genuinely has no name, use `targetPort: 8001` instead.

**3. Prometheus must be willing to look at your ServiceMonitor.** The operator only picks
up ServiceMonitors matching `spec.serviceMonitorSelector`. Our `prometheus-values.yaml`
sets `serviceMonitorSelectorNilUsesHelmValues: false`, which makes that selector nil and
therefore matches everything. On a stock install you would need the
`release: prometheus` label — which we add below anyway, out of habit.

In [ ]:
%%writefile ./servicemonitor.yaml
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: user-service-monitor
  namespace: k8s-lab
  labels:
    release: prometheus
spec:
  # Matches the LABELS ON THE SERVICE named user-service.
  selector:
    matchLabels:
      app: user-service
  endpoints:
    # `port` is the NAME of the Service port (`name: http` in service.yaml),
    # NOT the number 8001. A number here matches nothing and yields no targets.
    - port: http
      path: /metrics
      interval: 15s

In [ ]:
!kubectl apply -f ./servicemonitor.yaml
!kubectl get servicemonitor -n k8s-lab

# A ServiceMonitor that matches nothing produces ZERO targets and ZERO errors.
# The only way to know it worked is to ask Prometheus what it is scraping, so
# that is what we do -- this is the check the three bullet points below describe.
# Two separate things have to happen, and only the second one proves anything:
# the operator must turn the ServiceMonitor into scrape config (the target
# APPEARS, with health "unknown"), and Prometheus must then complete a scrape
# (health becomes "up"). Waiting only for the target to appear would pass on a
# ServiceMonitor pointing at an endpoint that never answers.
mine = []
deadline = time.time() + 160
while time.time() < deadline:
    active = prom_api("targets", state="active")["activeTargets"]
    mine = [t for t in active
            if t["labels"].get("namespace") == NS
            and t["labels"].get("service") == "user-service"]
    if mine and all(t["health"] == "up" for t in mine):
        break
    print(f"  targets: {len(mine)}  health: {[t['health'] for t in mine]}")
    time.sleep(10)

assert mine, (
    "the ServiceMonitor produced NO scrape targets after 150s. The three usual causes:\n"
    "  1. endpoints[].port is a number instead of a Service port NAME\n"
    "  2. spec.selector does not match the Service's labels\n"
    "     -> kubectl get svc user-service -n k8s-lab --show-labels\n"
    "  3. the Service has no endpoints at all\n"
    "     -> kubectl get endpoints user-service -n k8s-lab"
)
for t in mine:
    print(f"  {t['scrapeUrl']}  health={t['health']}  {t.get('lastError') or ''}")
assert all(t["health"] == "up" for t in mine), (
    "Prometheus discovered the targets but never completed a scrape: "
    f"{[(t['health'], t.get('lastError')) for t in mine]}"
)

# And the app's own series really landed in the TSDB.
series = prom_api("query", query='user_service_total_users{namespace="k8s-lab"}')["result"]
assert series, "the target is up but no user_service_* series exists yet"
print(f"\n✅ {len(mine)} targets scraped; user_service_total_users = "
      f"{series[0]['value'][1]} (the app has 3 sample users)")

In [ ]:
# Verify our app really exposes Prometheus-format text on /metrics.
# The image is python:3.12-slim -- no curl, no wget -- so use Python.
!kubectl exec -n k8s-lab deploy/user-service -- python -c "import urllib.request; print(urllib.request.urlopen('http://localhost:8001/metrics').read().decode())"

body = subprocess.run(
    ["kubectl", "exec", "-n", NS, "deploy/user-service", "--", "python", "-c",
     "import urllib.request;"
     "print(urllib.request.urlopen('http://localhost:8001/metrics').read().decode())"],
    capture_output=True, text=True, timeout=120).stdout

# The failure this guards against: FastAPI serialises a bare `str` return value
# as JSON, so the whole exposition would arrive wrapped in quotes with \n
# escapes, and Prometheus would reject it with "invalid metric name". The
# endpoint uses PlainTextResponse precisely to avoid that.
assert not body.lstrip().startswith('"'), (
    "/metrics is returning JSON, not the Prometheus text format -- Prometheus "
    "would reject it with 'invalid metric name'"
)
assert "# TYPE user_service_total_users gauge" in body, \
    f"missing the TYPE line Prometheus needs:\n{body[:300]}"
print("✅ /metrics is valid Prometheus exposition format")

## 📈 Step 5: Access Grafana Dashboards

Grafana is the visualization layer. The kube-prometheus-stack comes with dozens of
pre-built dashboards for Kubernetes.

Let's access it:

In [ ]:
# Port-forward Grafana, same free-port fallback as Prometheus above.
grafana_proc, grafana_port = start_port_forward(
    "svc/prometheus-grafana", 80, preferred=3000, ns="monitoring")

print(f"✅ Grafana available at: http://localhost:{grafana_port}")
print()
print("   Login credentials:")
print("   Username: admin")
print("   Password: admin  (set in prometheus-values.yaml)")
print()
print("   Try these dashboards (search in the dashboard menu):")
print("   1. 'Kubernetes / Compute Resources / Namespace (Pods)'")
print("   2. 'Kubernetes / Compute Resources / Node (Pods)'")
print("   3. 'Node Exporter / Nodes'")

with urllib.request.urlopen(f"http://localhost:{grafana_port}/api/health", timeout=15) as r:
    health = json.loads(r.read().decode())
assert health.get("database") == "ok", f"Grafana is not healthy: {health}"
print(f"\n(Grafana {health['version']} reports database: ok)")

### 🏠 Built-in Dashboards Worth Exploring

| Dashboard | What It Shows |
|-----------|---------------|
| Kubernetes / Compute Resources / Namespace | CPU, memory, network per namespace |
| Kubernetes / Compute Resources / Pod | CPU, memory, network per pod |
| Kubernetes / Networking / Namespace | Bandwidth, packet drops |
| Node Exporter / Nodes | OS-level: disk, CPU cores, memory, network |
| CoreDNS | DNS query rate, latency, errors |

## 🚨 Step 6: Create an Alert Rule

What good is monitoring if nobody gets notified when something breaks?

A **PrometheusRule** defines conditions that should trigger an alert.
Let's create one: "alert me if `user-service` has no available replicas for more than 2 minutes."

### Why not just `up == 0`?

`up` is the synthetic metric Prometheus writes for each **scrape target**, and the
obvious-looking rule is `up{namespace="k8s-lab"} == 0`. It is a trap, and it is worth
understanding before you rely on it in production.

`up` goes to `0` when Prometheus **has** a target and **fails to scrape it** — the pod
exists but refuses the connection. But when you scale a Deployment to zero, the Service
loses its endpoints, service discovery stops returning the target at all, and Prometheus
marks the series **stale**. A stale series is not `0`; it is *absent*. `up == 0` matches
nothing, so the alert never fires. The very outage you were alerting on makes the alert
disappear.

Two correct patterns:

| Goal | Expression |
|---|---|
| "this thing should have replicas and has none" | `kube_deployment_status_replicas_available{...} == 0` |
| "this target should exist and doesn't" | `absent(up{job="user-service"})` |

The first works because **kube-state-metrics** reports the *desired* state of the
Deployment object, which still exists at 0 replicas. That is the general lesson: alert on
a signal that survives the failure. We use both below.

In [ ]:
%%writefile ./alert-rule.yaml
apiVersion: monitoring.coreos.com/v1
kind: PrometheusRule
metadata:
  name: app-alerts
  namespace: monitoring
  labels:
    release: prometheus
spec:
  groups:
    - name: k8s-lab.rules
      rules:
        # Fires when the Deployment object still exists but has 0 available pods.
        # Survives scale-to-zero, unlike `up == 0`.
        - alert: DeploymentHasNoAvailableReplicas
          expr: kube_deployment_status_replicas_available{namespace="k8s-lab"} == 0
          for: 2m
          labels:
            severity: critical
          annotations:
            summary: "Deployment {{ $labels.deployment }} has no available replicas"
            description: "{{ $labels.deployment }} in namespace {{ $labels.namespace }} has had 0 available replicas for more than 2 minutes."

        # Fires when a target that SHOULD be scraped has vanished from service
        # discovery entirely. absent() returns 1 when its argument has no series.
        - alert: UserServiceTargetMissing
          expr: absent(up{namespace="k8s-lab", service="user-service"})
          for: 2m
          labels:
            severity: warning
          annotations:
            summary: "Prometheus has no scrape target for user-service"
            description: "The user-service ServiceMonitor produced no targets for 2 minutes."

        # The classic one, kept for contrast: this fires only when the pod is up
        # but the scrape FAILS. It will NOT fire during the scale-to-zero test.
        - alert: ScrapeFailing
          expr: up{namespace="k8s-lab"} == 0
          for: 2m
          labels:
            severity: warning
          annotations:
            summary: "Prometheus cannot scrape {{ $labels.job }}"

In [ ]:
!kubectl apply -f ./alert-rule.yaml
!kubectl get prometheusrule -n monitoring app-alerts

# The PrometheusRule object existing is not the same as Prometheus having LOADED
# it -- the operator has to notice the CR, rewrite the rule files into the
# Prometheus pod, and trigger a reload. Wait for the rules to show up in
# Prometheus's own API before the test below depends on them.


def loaded_rules():
    groups = prom_api("rules")["groups"]
    for g in groups:
        if g["name"] == "k8s-lab.rules":
            return {r["name"]: r for r in g["rules"]}
    return {}


# A freshly loaded alerting rule reports state "unknown" until Prometheus has
# evaluated it once (every 30s by default). "unknown" is not "inactive" and it is
# certainly not "firing" -- so wait for the first evaluation before judging.
rules = {}
deadline = time.time() + 160
while time.time() < deadline:
    rules = loaded_rules()
    states = {k: v["state"] for k, v in rules.items()}
    if len(rules) == 3 and "unknown" not in states.values():
        break
    print("  ", states or "rules not loaded yet")
    time.sleep(10)

assert len(rules) == 3, \
    f"Prometheus loaded {len(rules)} of the 3 rules after 160s: {sorted(rules)}"
states = {k: v["state"] for k, v in rules.items()}
assert all(s == "inactive" for s in states.values()), \
    f"nothing is broken yet, so every alert should be inactive: {states}"
print(f"\n✅ Prometheus loaded {sorted(rules)}, all inactive")

### 🧪 Exercise: Trigger the Alert

Let's deliberately break something and see the alert fire:

1. Scale user-service to 0 replicas
2. Wait ~2 minutes
3. Check the alert status in Prometheus
4. Fix it by scaling back up

In [ ]:
# Break it! Scale to zero replicas
!kubectl scale deployment user-service -n k8s-lab --replicas=0

print("\n💥 user-service scaled to 0 — no pods running!")
print()
print("   Expected once the `for: 2m` window elapses:")
print("     DeploymentHasNoAvailableReplicas -> PENDING, then FIRING   ✅")
print("     UserServiceTargetMissing         -> PENDING, then FIRING   ✅")
print("     ScrapeFailing (up == 0)          -> stays INACTIVE         ⚠️  <- the lesson")
print()

# kube-state-metrics is scraped every 30s and the rules have `for: 2m`, so the
# earliest anything can fire is a bit over two minutes from now. Wait most of
# that here; the next cell does the polling and the asserting.
for elapsed in range(0, 120, 20):
    states = {name: r["state"] for name, r in loaded_rules().items()}
    print(f"  t+{elapsed:>3}s  {states}")
    time.sleep(20)

In [ ]:
# Now the actual verification. Two of these three alerts must fire and the third
# must NOT -- that asymmetry is the entire point of the section above, and it is
# worth failing the notebook over if it ever stops being true.
deadline = time.time() + 170
while time.time() < deadline:
    states = {name: r["state"] for name, r in loaded_rules().items()}
    if states.get("DeploymentHasNoAvailableReplicas") == "firing" and \
            states.get("UserServiceTargetMissing") == "firing":
        break
    print(f"  waiting... {states}")
    time.sleep(10)

print("\nfinal states:", states)

assert states["DeploymentHasNoAvailableReplicas"] == "firing", (
    "kube_deployment_status_replicas_available == 0 should fire: the Deployment "
    "object still exists at 0 replicas, so kube-state-metrics still reports it."
)
assert states["UserServiceTargetMissing"] == "firing", \
    "absent(up{...}) should fire once service discovery drops the target"
assert states["ScrapeFailing"] == "inactive", (
    "up == 0 FIRED -- that would break the lesson. It is supposed to stay inactive: "
    "scaling to zero removes the target from service discovery entirely, so the `up` "
    "series goes stale rather than to 0, and `up == 0` matches nothing."
)

# The same point, straight from PromQL:
assert prom_api("query", query='up{namespace="k8s-lab", service="user-service"}')["result"] == [], \
    "expected the `up` series to be absent, not zero"
assert prom_api("query", query='absent(up{namespace="k8s-lab", service="user-service"})')["result"], \
    "absent() should return 1 for a vanished target"

print("\n✅ the outage fired the two alerts that survive it, and NOT `up == 0`")

In [ ]:
# Fix it! Scale back up
!kubectl scale deployment user-service -n k8s-lab --replicas=2
!kubectl wait --for=condition=ready pod -l app=user-service -n k8s-lab --timeout=120s

# Alerts have no `for` window on the way down: as soon as the expression stops
# matching, the next rule evaluation clears them. Confirm they really resolve --
# an alert that fires and never clears is its own kind of incident.
deadline = time.time() + 170
while time.time() < deadline:
    states = {name: r["state"] for name, r in loaded_rules().items()}
    if all(s == "inactive" for s in states.values()):
        break
    print(f"  resolving... {states}")
    time.sleep(10)

assert all(s == "inactive" for s in states.values()), \
    f"alerts did not clear after the service recovered: {states}"
print("\n✅ user-service is back and every alert auto-resolved:", states)

## 🧹 Clean Up

The monitoring stack uses significant resources. Remove it if you need to free up memory for later labs.

In [ ]:
# Stop the port-forwards this notebook started.
for _name in ("prom_proc", "grafana_proc"):
    _obj = globals().get(_name)
    if _obj is not None:
        _obj.terminate()

# Restore the replica count the alert test changed.
!kubectl scale deployment user-service -n k8s-lab --replicas=2

# Clean up generated files
!rm -f ./servicemonitor.yaml ./alert-rule.yaml

# Remove the monitoring stack. This is not tidiness -- it is capacity planning.
# kube-prometheus-stack is the heaviest thing in this series (Prometheus alone
# requests 400 Mi and grows as it collects), notebook 08 installs Istio plus a
# sidecar on every pod, and on a 6 GB cluster the two together push pods into
# Pending. Nothing after this notebook reads from Prometheus -- notebook 10's HPA
# uses metrics-server, which is a completely separate pipeline (see the table
# above) and stays installed.
#
# Want to keep exploring Grafana instead? Comment these two lines out; just be
# ready to run them before notebook 08. Reinstalling is the same one-liner from
# the top of this notebook.
!helm uninstall prometheus -n monitoring --ignore-not-found || true
!kubectl delete namespace monitoring --ignore-not-found

# metrics-server must survive -- notebook 10 depends on it.
import subprocess
assert subprocess.run(["kubectl", "get", "apiservice", "v1beta1.metrics.k8s.io"],
                      capture_output=True).returncode == 0, \
    "metrics-server was removed too; notebook 10's HPA needs metrics.k8s.io"
assert subprocess.run(["kubectl", "get", "namespace", "monitoring"],
                      capture_output=True).returncode != 0, \
    "the monitoring namespace is still present"
print("✅ Cleaned up. metrics-server stays; the Prometheus stack is gone so notebook 08"
      "\n   has room for Istio.")

## 🎓 What You Learned

In this notebook you:

1. **Enabled Metrics Server** and used `kubectl top` to see CPU and memory usage
2. **Installed Prometheus + Grafana** using the kube-prometheus-stack Helm chart
3. **Queried Prometheus** with PromQL for built-in Kubernetes metrics
4. **Created a ServiceMonitor** to scrape your application's custom `/metrics` endpoint
5. **Explored Grafana dashboards** for cluster-wide and namespace-level observability
6. **Created a PrometheusRule** alert that fires when a service goes down
7. **Triggered and resolved** an alert by scaling a deployment to zero and back

### Key Takeaways

- **Metrics Server** is for `kubectl top` and HPA — it has no history
- **Prometheus** stores time-series data and is the backbone of Kubernetes monitoring
- **ServiceMonitor** is how you tell Prometheus to scrape your app — no config file editing
- **Grafana** comes with dozens of pre-built dashboards for Kubernetes
- **PrometheusRule** defines alerting conditions — always create alerts for critical services
- **`up == 0` does not detect a scaled-to-zero service.** Alert on a signal that
  survives the failure (`kube_deployment_status_replicas_available`, or `absent()`)
- A ServiceMonitor's `endpoints[].port` is a Service port **name**; a number there
  produces zero targets and zero errors

### Next Steps

In **Notebook 06** you'll learn how to secure your cluster with RBAC, NetworkPolicies,
and Pod Security Standards — so different teams can safely share the same cluster.